# Lesson 4: Setting realistic Goals using AI

Welcome back to another exciting lesson! Today we will be learning more about modern Generative AI and use large language models to create a cool feature for our Priority Bank: A cost estimator.

> 👯 **Work in pairs!** One person types (the **Driver**), the other guides (the **Navigator**). Switch roles after each exercise.

## Exercises
1. Learn what LLMs are
2. Learn how to embed LLMs into your code
3. Learn how to add tool-calling to LLMs
4. Develop system prompt for Priority Bank Cost Estimation Feature

Today's goal is to add a new feature that can make adding goals to our Priority Bank even more convenient and realistic. Sometimes we can become aware of a financial goal we want to achieve, but it can be hard to figure out exactly how much it is going to cost.

In those moments, it can be a really good idea to spent some time researching this on information on the internet. However, with the advert of modern GenAI, we don't need to do this ourself. Instead, we can create a research assistent using code and LLMs that can do the task for us, so we can focus on making the financial decisions.

In [33]:
import os
import json
from dotenv import load_dotenv

load_dotenv()

from mistralai import Mistral
from tavily import TavilyClient
from pydantic import BaseModel
from pydantic import Field

---
## Exercise 1: What are LLMs
#@TODO: Write this and brainstorm some simple exercises

---
## Exercise 2: Call an LLM
#@TODO: Write some intro

In [2]:
client = Mistral(api_key=os.getenv("MISTRAL_KEY"))
model = "mistral-small-2506"

In [3]:
your_message = "What is number one best financial advice you can give a 13-16 y/o girl?"

chat_response = client.chat.complete(
    model= model,
    messages = [
        # TODO: Fill out
        {"role":"user",  "content": your_message}
    ],
)

print("------------------ USER ------------------")
print(your_message)
print("------------------ LLM ------------------")
print(chat_response.choices[0].message.content)

------------------ USER ------------------
What is number one best financial advice you can give a 13-16 y/o girl?
------------------ LLM ------------------
The **#1 best financial advice** I can give a 13–16-year-old girl is:

**"Start building good money habits NOW—because small, smart choices today will set you up for a lifetime of financial freedom."**

### **Here’s how to do it:**
1. **Learn the basics** – Understand how money works (saving, budgeting, interest, credit, investing).
2. **Save first, spend later** – Pay yourself first (even if it’s just $5 a week). Use the **50/30/20 rule** (50% needs, 30% wants, 20% savings).
3. **Avoid debt traps** – Credit cards and loans can be dangerous if misused. Only borrow what you can repay.
4. **Start investing early** – Even small amounts in a **Roth IRA** (if you have earned income) or a low-cost index fund can grow into big wealth over time.
5. **Earn more** – Side hustles (babysitting, tutoring, selling crafts) teach responsibility an

### The importance of system prompts
Two calls - one with and one without system prompt

In [12]:
system_prompt = "Only provide short answers that rhyme."

chat_response = client.chat.complete(
    model= model,
    messages = [
        # TODO: Fill out
        {"role":"system", "content": system_prompt},
        {"role":"user",  "content": your_message}
    ],
)

print("------------------ USER ------------------")
print(your_message)
print("------------------ LLM ------------------")
print(chat_response.choices[0].message.content)

------------------ USER ------------------
What is number one best financial advice you can give a 13-16 y/o girl?
------------------ LLM ------------------
Save and invest, don't just spend,
Your future self will thank you in the end.


---
## Exercise 3: Tool-calling LLMs
#@TODO: write this

In [86]:
def search_tavily(search_query: str) -> list:
    """Search for specific query on the internet."""
    client = TavilyClient(api_key=os.getenv("TAVILY_KEY"))
    result = client.search(query=search_query)
    return result.get("results", [])

In [80]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_tavily",
            "description": "Search for specific query on the internet.",
            "parameters": {
                "type": "object",
                "properties": {
                    "search_query": {
                        "type": "string",
                        "description": "The search query to search for online.",
                    }
                },
                "required": ["search_query"],
            },
        },
    }
]

In [ ]:
system_prompt = (
    "You are an expert in finding and estimating the cost of an item or financial goal based on limited user input. "
    "You will receive a few words from the user on what they are setting as a new financial goal for themselves and "
    "your job will be to give them an estimate for how much it will cost them to achieve this goal / aquire the item.\n"
    "You are welcome to use the search tool before giving your answer, but focus on finding pricing information for the item.\n"
    "The users are 13-16 year old danish girls, so provide cost estimates in DKK and makes sure to adapt any web searches "
    "to fit with what a teenage girl would like to do (i.e. if the user says 'New clothes', search 'new clothes for teenage girls price').\n"   
    "Your response should ONLY be a number in DKK. No explanation or anything else."
)

new_goal = "New Adidas Running Shoes"

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": new_goal},
]

# 1. Call LLM
chat_response = client.chat.complete(
    model= model,
    messages = messages,
    tools = tools,
    temperature=1
)

messages.append(chat_response.choices[0].message)
response = chat_response.choices[0].message

# 2.1 Execute tool calls
for tool_call in response.tool_calls:
    args = tool_call.function.arguments
    if tool_call.function.name == 'search_tavily':
        web_results = search_tavily(**json.loads(args))
        tool_result = "\n\n".join(f"{r['title']}\n{r['content']}" for r in web_results)
        messages.append({
            "role":"tool",
            "name":tool_call.function.name,
            "content": tool_result,
            "tool_call_id":tool_call.id
        })

In [91]:
for message in messages:
    if isinstance(message, dict):
        print(f"------- {message['role']} ------")
        print(message['content'])
        print()
    else:
        if message.tool_calls:
            for tool_call in message.tool_calls:
                print(f"------- tool_call ------")
                print(tool_call.function.name)
                print(tool_call.function.arguments)
                print()
        else:
            print(f"------- agent ------")
            print(message.content)
            print()

------- system ------
You are an expert in finding and estimating the cost of an item or financial goal based on limited user input. You will receive a few words from the user on what they are setting as a new financial goal for themselves and your job will be to give them an estimate for how much it will cost them to achieve this goal / aquire the item.
You are welcome to use the search tool before giving your answer, but focus on finding pricing information for the item.
The users are 13-16 year old danish girls, so provide cost estimates in DKK and makes sure to adapt any web searches to fit with what a teenage girl would like to do (i.e. if the user says 'New clothes', search 'new clothes for teenage girls price').
Your response should ONLY be a number in DKK. No explanation or anything else.

------- user ------
New Adidas Running Shoes

------- tool_call ------
search_tavily
{"search_query": "New Adidas Running Shoes for teenage girls price"}

------- tool ------
adidas Girls' Ru

Let's call the final model to get the answer

In [92]:
# 3. Get final response from LLM
chat_response = client.chat.complete(
    model= model,
    messages = messages,
    tools = tools,
    temperature=1
)

chat_response.choices[0].message.content

'499'

### The importance of structured output
While this 

In [93]:
class Output(BaseModel):
    estimated_price: float = Field(
        description="The estimated price for the item provided by the user."
    )
    reasoning: str = Field(
        description="A short reasoning, describing the evidence for this price."
    )

In [96]:
# 3. Get structured response from LLM
chat_response = client.chat.parse(
    model= model,
    messages = messages,
    tools = tools,
    response_format=Output
)

structured_output = json.loads(chat_response.choices[0].message.content)
structured_output

{'estimated_price': 400,
 'reasoning': 'The prices for new Adidas running shoes for teenage girls vary depending on the specific model and where you purchase them. On average, the prices range from around $49.99 to $120. Converting these prices to DKK using an approximate exchange rate of 1 USD to 6.5 DKK, we get the following estimates: $49.99 is approximately 325 DKK and $120 is approximately 780 DKK. Therefore, the average price for new Adidas running shoes for teenage girls in DKK is around 400 DKK.'}

---
## Exercise 4: Putting it all together
Let's create the function for estimating the price of a new 

In [99]:
def estimate_price(user_input: str, system_prompt:str, model: str = "mistral-small-2506") -> dict:
    """Use an LLM to search the internet and estimate an appropriate price for a goal."""

    # 1. Define tools available to the LLM
    tools = [{
        "type": "function",
        "function": {
            "name": "search_tavily",
            "description": "Search for specific query on the internet.",
            "parameters": {
                "type": "object",
                "properties": {
                    "search_query": {
                        "type": "string",
                        "description": "The search query to search for online.",
                    }
                },
                "required": ["search_query"],
            },
        },
    }]

    # 2. Create chat client
    client = Mistral(api_key=os.getenv("MISTRAL_KEY"))

    # 3. Define messages
    messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_input},
    ]

    # 4. Call LLM to get search result
    chat_response = client.chat.complete(
        model= model,
        messages = messages,
        tools = tools,
        temperature=1
    )
    response = chat_response.choices[0].message

    messages.append(response)
    
    # 5. Execute tool calls
    if response.tool_calls:
        for tool_call in response.tool_calls:
            args = tool_call.function.arguments
            if tool_call.function.name == 'search_tavily':
                web_results = search_tavily(**json.loads(args))
                tool_result = "\n\n".join(f"{r['title']}\n{r['content']}" for r in web_results)
                messages.append({
                    "role":"tool",
                    "name":tool_call.function.name,
                    "content": tool_result,
                    "tool_call_id":tool_call.id
                })
    else:
        return {"estimated_price": response.content, "reasoning": "LLM estimated immediately."}
    
    # 6. Get structured response from LLM
    chat_response = client.chat.parse(
        model= model,
        messages = messages,
        tools = tools,
        response_format=Output
    )

    structured_output = json.loads(chat_response.choices[0].message.content)

    return structured_output

In [100]:
out = estimate_price(user_input = "Nye nike sneakers", system_prompt=system_prompt)
out

{'estimated_price': 749.9,
 'reasoning': 'The estimated price for Nike sneakers for teenage girls is approximately 749.9 DKK. This estimate is based on the search results showing various Nike sneaker models priced around this range, such as the Nike Dunk Low and Nike Air Max 95 Recraft, which are popular among teenagers.'}